# IEX Simulation: Interval Comparison + Actual vs Planned Audit
**Excel (5 sheets):** VNT Productive | VNT Heads | PST Productive | PST Heads | Detail  
**Productive = Heads / 2** (30-min interval = 0.5h), 1 decimal place  
**Audit:** Compare IEX_ACTUAL_ADJUSTMENT vs IEX_AFTER_ADJUSTMENT post-swap

In [1]:
import time
from datetime import datetime
import pandas as pd
import numpy as np
import os
import pathlib
from IPython.display import display, HTML
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

In [2]:
WEEK_MONDAY = '2026-08-31'

_week    = pd.Timestamp(WEEK_MONDAY)
_week_end = _week + pd.Timedelta(days=6)
print(f"Week: {_week.strftime('%A %d %b %Y')} to {_week_end.strftime('%A %d %b %Y')}")

Week: Monday 31 Aug 2026 to Sunday 06 Sep 2026


In [3]:
_home = os.path.expanduser('~').replace('\\', '/')
_base = f'{_home}/Concentrix Corporation/WFM-Expedia-HCM - Branding files'
_sim  = f'{_base}/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION'

PATHS = {
    'structured_v1'              : f'{_sim}/IEX_STRUCTURED_V1',
    'after_adjustment'           : f'{_sim}/IEX_AFTER_ADJUSTMENT',
    'actual_adjustment'          : f'{_sim}/IEX_ACTUAL_ADJUSTMENT',
    'intervals_original'         : f'{_sim}/IEX_INTERVALS_ORIGINAL',
    'intervals_after_adjustment' : f'{_sim}/IEX_INTERVALS_AFTER_ADJUSTMENT',
    'excel_output'               : _sim,
}
for k, v in PATHS.items():
    os.makedirs(v, exist_ok=True)
    print(f"  OK  {k}")

ALL_VNT_INTERVALS = [
    f"{h:02d}:{m:02d}-{(h+(m+30)//60)%24:02d}:{(m+30)%60:02d}"
    for h in range(24) for m in (0, 30)
]
print("Interval grid:", len(ALL_VNT_INTERVALS), "slots (00:00-00:30 to 23:30-00:00)")

  OK  structured_v1
  OK  after_adjustment
  OK  actual_adjustment
  OK  intervals_original
  OK  intervals_after_adjustment
  OK  excel_output
Interval grid: 48 slots (00:00-00:30 to 23:30-00:00)


In [4]:
SCHEDULED_ACTIVITIES = {
    'Open Time','Extra Hours','No Call/No Show','PTO','Training Offline',
    'Sick Leave','Paid Leave','Termination','Off Phone Misc',
    'Billable Training','Nesting Training','Training',
}

def convert_to_time(df, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], format='%I:%M %p', errors='coerce').dt.time
    return df

def create_datetime_vec(df):
    base = pd.to_datetime(df['Date'])
    def _c(b, t): return pd.to_datetime(b.astype(str)+' '+t.astype(str), errors='coerce')
    s = _c(base, df['Start_Action']); e = _c(base, df['End_Action']); sh = _c(base, df['Start_Shift'])
    s += pd.to_timedelta(((df['Scheduled']!=0)&s.notna()&sh.notna()&(s<sh)).astype(int), unit='D')
    e += pd.to_timedelta(((df['Scheduled']!=0)&e.notna()&s.notna()&(e<s)).astype(int), unit='D')
    return s.where(df['Scheduled']!=0, pd.NaT), e.where(df['Scheduled']!=0, pd.NaT)

def _shift_range(df, mask, prefix):
    agg = (df[mask].groupby(['Date','IEX_ID'],sort=False)
           .agg(**{f'Datetime_{prefix}_Start_Shift':('Datetime_Start_Action','min'),
                  f'Datetime_{prefix}_End_Shift'  :('Datetime_End_Action','max')})
           .reset_index())
    sc,ec = f'Datetime_{prefix}_Start_Shift', f'Datetime_{prefix}_End_Shift'
    agg[f'{prefix} Shift'] = agg[sc].dt.strftime('%H%M')+'-'+agg[ec].dt.strftime('%H%M')
    return agg

def build_sorted_df(csv_path):
    df = pd.read_csv(csv_path, dtype=str)
    df['Date']   = pd.to_datetime(df['Date'], errors='coerce')
    df['IEX_ID'] = df['Agent'].str.extract(r'(\d+)', expand=False).astype('Int64')
    df['Month']       = df['Date'].dt.strftime('%b-%y')
    df['Week_Monday'] = (df['Date']-pd.to_timedelta(df['Date'].dt.dayofweek,unit='D')).dt.normalize()
    df['Agent Name']  = df['Agent'].str.extract(r'\d+ (.+)', expand=False).str.upper()
    df['Scheduled']   = df.groupby(['Date','IEX_ID'])['Scheduled Activity'].transform(
        lambda x: 1 if x.isin(SCHEDULED_ACTIVITIES).any() else 0)
    if 'Generate Date' in df.columns:
        df['Generate Date'] = pd.to_datetime(df['Generate Date'], errors='coerce')
        mg = df.groupby(['IEX_ID','Date'],sort=False)['Generate Date'].max().reset_index(name='_mg')
        df = df.merge(mg,on=['IEX_ID','Date'],how='left')
        df = df[df['Generate Date']==df['_mg']].drop(columns=['_mg'])
    df = convert_to_time(df,['Start_Shift','End_Shift','Start_Action','End_Action'])
    df['Datetime_Start_Action'], df['Datetime_End_Action'] = create_datetime_vec(df)
    df['_tr'] = df['Scheduled Activity'].str.contains('training',case=False,na=False)
    otk = set(df.loc[df['Scheduled Activity']=='Open Time',['Date','IEX_ID']].itertuples(index=False,name=None))
    df['_ho'] = [(d,i) in otk for d,i in zip(df['Date'],df['IEX_ID'])]
    fl = df['Scheduled Activity'].isin(['Open Time','Extra Hours','No Call/No Show','System Outage','Offline'])|df['_tr']
    fi = (df['_ho']&(df['Scheduled Activity'].isin(['Open Time','No Call/No Show','System Outage','Offline'])|df['_tr']))|~df['_ho']
    df = (df.merge(_shift_range(df,fl,'Fluctuate'),on=['Date','IEX_ID'],how='left')
            .merge(_shift_range(df,fi,'First'),on=['Date','IEX_ID'],how='left'))
    df = df[~(df['Scheduled Activity'].isin(['Lunch','Break'])&df['Fluctuate Shift'].isna())].copy()
    df['Duration'] = (df['Datetime_End_Action']-df['Datetime_Start_Action']).dt.total_seconds()
    df['Time_Of_Day'] = (df['Datetime_First_End_Shift']-df['Datetime_First_Start_Shift']).dt.total_seconds()/3600
    df = df.sort_values(['Date','IEX_ID','Datetime_Start_Action'],na_position='last').drop_duplicates()
    TRACT=['Training Offline','Training','Training Nesting','Nesting Training','Billable Training']
    AMAP={'Open Time':'Open Time','Break':'Break Time','Lunch':'Lunch Time','Extra Hours':'Extra Time','No Call/No Show':'NCNS','PTO':'AL'}
    tr=(df[df['Scheduled Activity'].isin(TRACT)].groupby(['Date','IEX_ID'],sort=False)['Duration'].sum().div(3600).reset_index(name='Training'))
    df=df.merge(tr,on=['Date','IEX_ID'],how='left')
    tt=(df[df['Scheduled Activity'].isin(AMAP)].groupby(['Date','IEX_ID','Scheduled Activity'],sort=False)['Duration']
        .sum().div(3600).reset_index().assign(col_name=lambda d:d['Scheduled Activity'].map(AMAP))
        .pivot_table(index=['Date','IEX_ID'],columns='col_name',values='Duration',aggfunc='sum').reset_index())
    tt.columns.name=None; df=df.merge(tt,on=['Date','IEX_ID'],how='left')
    df=df.sort_values(['Date','IEX_ID','Datetime_Start_Action'],na_position='last')
    df['FSA']=df.groupby(['Date','IEX_ID'])['Scheduled Activity'].transform('first')
    fsa=df['FSA']; ot=df.get('Open Time',pd.Series(np.nan,index=df.index))
    et=df.get('Extra Time',pd.Series(np.nan,index=df.index)); nc=df.get('NCNS',pd.Series(0,index=df.index))
    conds=[fsa.isin(['Holiday','Bereavement','Off','Off Phone Misc','Unscheduled']),fsa=='PTO',
           fsa.isin(['Sickness','Sick Leave']),fsa.isin(['Training Offline','Billable Training','Nesting Training']),
           fsa.isin(['Paid Leave']),fsa.isin(['Leave']),fsa.isin(['Termination']),
           (fsa=='Extra Hours')&ot.isna(),(ot>0)&(nc>0)&(nc<=5),
           ((ot==0)|ot.isna())&(nc>5),(ot>5)&(et>0),(ot>0)&(ot<5)&(et>0),(ot==0)&(et>0),(ot>0)]
    choices=[fsa,'AL','SL','Training Offline','CO','LWP','Termination','PO','HDL','NCNS','PR - OT','HDL - OT','PO','PR']
    df['Shift Tracking']=np.select(conds,choices,default=fsa)
    df['First Shift']=np.where(df['Time_Of_Day'].isna(),df['Scheduled Activity'],df['First Shift'])
    return df.drop(columns=['_tr','_ho','FSA'],errors='ignore')

def generate_intervals(df_input):
    df=df_input.copy()
    df['Datetime_Start_Action']=pd.to_datetime(df['Datetime_Start_Action'],errors='coerce')
    df['Datetime_End_Action']  =pd.to_datetime(df['Datetime_End_Action'],  errors='coerce')
    df=df.dropna(subset=['Datetime_Start_Action','Datetime_End_Action'])
    ov=df['Datetime_End_Action']<df['Datetime_Start_Action']
    df.loc[ov,'Datetime_End_Action']+=pd.Timedelta(days=1)
    df['_fl']=df['Datetime_Start_Action'].dt.floor('30min')
    df['_ce']=df['Datetime_End_Action'].dt.ceil('30min')
    df['_n']=(((df['_ce']-df['_fl'])/pd.Timedelta(minutes=30)).fillna(0).astype(int))
    df=df[df['_n']>0].copy()
    e=df.loc[df.index.repeat(df['_n'])].copy()
    e['_i']=e.groupby(level=0).cumcount()
    e['Datetime_Start_Time_Full']=e['_fl']+pd.to_timedelta(e['_i']*30,unit='m')
    e['Datetime_End_Time_Full']  =e['Datetime_Start_Time_Full']+pd.Timedelta(minutes=30)
    e['Datetime_Start_Time']=np.maximum(e['Datetime_Start_Time_Full'],e['Datetime_Start_Action'])
    e['Datetime_End_Time']  =np.minimum(e['Datetime_End_Time_Full'],  e['Datetime_End_Action'])
    e=e[e['Datetime_Start_Time']<e['Datetime_End_Time']].copy()
    e['Duration']=(e['Datetime_End_Time']-e['Datetime_Start_Time']).dt.total_seconds()/3600
    e['VNT_Intervals']=e['Datetime_Start_Time_Full']
    e['PST_Intervals']=(e['VNT_Intervals'].dt.tz_localize('Asia/Ho_Chi_Minh').dt.tz_convert('America/Los_Angeles').dt.tz_localize(None))
    vs=e['VNT_Intervals'].dt.strftime('%H:%M'); ve=(e['VNT_Intervals']+pd.Timedelta(minutes=30)).dt.strftime('%H:%M')
    ps=e['PST_Intervals'].dt.strftime('%H:%M'); pe=(e['PST_Intervals']+pd.Timedelta(minutes=30)).dt.strftime('%H:%M')
    e['VNT_Interval_Range']=vs+'-'+ve; e['PST_Interval_Range']=ps+'-'+pe
    e['Work Category']=np.where(e['Scheduled Activity'].isin(['Open Time','Extra Hours']),'Productive','Unproductive')
    e=e.rename(columns={'Date':'Date_Converted','IEX_ID':'IEX ID'})
    COLS=['Month','Week_Monday','Date_Converted','Agent Name','IEX ID','First Shift',
          'Scheduled Activity','VNT_Intervals','PST_Intervals','VNT_Interval_Range','PST_Interval_Range',
          'Datetime_Start_Time','Datetime_End_Time','Duration','Work Category']
    return e[[c for c in COLS if c in e.columns]].reset_index(drop=True)

def run_pipeline(src_folder, out_folder, label, week_monday):
    ws=week_monday.strftime('%Y_%m_%d')
    src=next((pathlib.Path(src_folder)/n for n in [f'IEX_Structured_{ws}.csv',f'IEX_Adjusted_{ws}.csv']
              if (pathlib.Path(src_folder)/n).exists()), None)
    if src is None: print(f"[{label}] No file for week {ws}"); return pd.DataFrame()
    print(f"[{label}] Reading: {src.name}")
    sd=build_sorted_df(str(src)); iv=generate_intervals(sd)
    out=pathlib.Path(out_folder)/f'{week_monday.strftime("%Y-%m-%d")}.csv'
    iv.to_csv(out,index=False,encoding='utf-8-sig')
    print(f"[{label}] {len(iv):,} interval rows saved")
    return iv

print("All helper functions loaded.")

All helper functions loaded.


In [5]:
print("Processing ORIGINAL ...")
iv_original = run_pipeline(PATHS['structured_v1'], PATHS['intervals_original'], 'ORIGINAL', _week)
print("\nProcessing ADJUSTED ...")
iv_adjusted = run_pipeline(PATHS['after_adjustment'], PATHS['intervals_after_adjustment'], 'ADJUSTED', _week)

def _load(folder, week):
    p = pathlib.Path(folder)/f'{week.strftime("%Y-%m-%d")}.csv'
    if not p.exists(): return pd.DataFrame()
    df = pd.read_csv(p)
    df['Date_Converted'] = pd.to_datetime(df['Date_Converted'], errors='coerce')
    return df

if iv_original.empty: iv_original = _load(PATHS['intervals_original'], _week)
if iv_adjusted.empty: iv_adjusted = _load(PATHS['intervals_after_adjustment'], _week)
print(f"\nORIGINAL: {len(iv_original):,} | ADJUSTED: {len(iv_adjusted):,}")

Processing ORIGINAL ...
[ORIGINAL] Reading: IEX_Structured_2026_08_31.csv
[ORIGINAL] 11,360 interval rows saved

Processing ADJUSTED ...
[ADJUSTED] Reading: IEX_Adjusted_2026_08_31.csv
[ADJUSTED] 11,252 interval rows saved

ORIGINAL: 11,360 | ADJUSTED: 11,252


In [6]:
def build_comparison(orig, adj):
    G=['Date_Converted','VNT_Interval_Range','PST_Interval_Range']
    po=(orig[orig['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='Productive_Before'))
    pa=(adj[adj['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='Productive_After'))
    cmp=po.merge(pa,on=G,how='outer').fillna(0)
    cmp['Productive_Delta']=cmp['Productive_After']-cmp['Productive_Before']
    cmp['Has_Change']=cmp['Productive_Delta'].abs()>0.001
    ao=orig[orig['Work Category']=='Productive'][['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name','First Shift']].rename(columns={'First Shift':'Shift_Before'})
    aa=adj[adj['Work Category']=='Productive'][['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name','First Shift']].rename(columns={'First Shift':'Shift_After'})
    ag=ao.merge(aa,on=['Date_Converted','VNT_Interval_Range','IEX ID','Agent Name'],how='outer')
    def _fmt(df,tag):
        sc='Shift_Before' if tag=='OUT' else 'Shift_After'
        df=df.copy(); df['d']=df['Agent Name'].fillna('?')+'('+df[sc].fillna('-')+')['+tag+']'
        return df.groupby(['Date_Converted','VNT_Interval_Range'])['d'].apply('; '.join).reset_index(name='_d')
    ch=pd.DataFrame()
    dp2=ag[ag['Shift_After'].isna()&ag['Shift_Before'].notna()]
    gn=ag[ag['Shift_Before'].isna()&ag['Shift_After'].notna()]
    if not dp2.empty: ch=pd.concat([ch,_fmt(dp2,'OUT')],ignore_index=True)
    if not gn.empty:  ch=pd.concat([ch,_fmt(gn,'IN')], ignore_index=True)
    if not ch.empty:
        ch=ch.groupby(['Date_Converted','VNT_Interval_Range'])['_d'].apply('; '.join).reset_index(name='Changed_Agents')
        cmp=cmp.merge(ch,on=['Date_Converted','VNT_Interval_Range'],how='left')
    else:
        cmp['Changed_Agents']=''
    cmp['Changed_Agents']=cmp['Changed_Agents'].fillna('')
    return cmp.sort_values(['Date_Converted','VNT_Interval_Range']).reset_index(drop=True)

comparison=build_comparison(iv_original,iv_adjusted)
print(f'Comparison: {len(comparison):,} rows | {comparison["Has_Change"].sum()} changed')
print('Productive_Before sample:', comparison['Productive_Before'].describe().round(2).to_dict())

Comparison: 336 rows | 165 changed
Productive_Before sample: {'count': 336.0, 'mean': 9.5, 'std': 4.45, 'min': 1.5, '25%': 6.0, '50%': 8.5, '75%': 13.0, 'max': 22.0}


In [7]:
def build_interval_pivot(orig, adj, week_dates, interval_col, all_ivls):
    G=['Date_Converted',interval_col]
    oc=(orig[orig['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='Before'))
    ac=(adj[adj['Work Category']=='Productive'].groupby(G,sort=False)['Duration'].sum().reset_index(name='After'))
    oc['DL']=oc['Date_Converted'].dt.strftime('%a\n%d/%m')
    ac['DL']=ac['Date_Converted'].dt.strftime('%a\n%d/%m')
    pvb=oc.pivot_table(index=interval_col,columns='DL',values='Before',aggfunc='sum',fill_value=0).reindex(all_ivls,fill_value=0)
    pva=ac.pivot_table(index=interval_col,columns='DL',values='After', aggfunc='sum',fill_value=0).reindex(all_ivls,fill_value=0)
    return pvb,pva

week_dates=sorted(pd.date_range(_week,periods=7,freq='D').normalize().tolist())
vnt_b,vnt_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'VNT_Interval_Range',ALL_VNT_INTERVALS)

ALL_PST_INTERVALS_ACTUAL=sorted(
    set(iv_original['PST_Interval_Range'].dropna())|set(iv_adjusted['PST_Interval_Range'].dropna()),
    key=lambda x:x.split('-')[0] if x else '')
pst_b,pst_a=build_interval_pivot(iv_original,iv_adjusted,week_dates,'PST_Interval_Range',ALL_PST_INTERVALS_ACTUAL)
print('VNT pivot:',vnt_b.shape,'| PST pivot:',pst_b.shape)
print('VNT Before sample (first 3 intervals):\n', vnt_b.iloc[:3].to_string())

VNT pivot: (48, 7) | PST pivot: (48, 7)
VNT Before sample (first 3 intervals):
 DL                  Fri\n04/09  Mon\n31/08  Sat\n05/09  Sun\n06/09  Thu\n03/09  Tue\n01/09  Wed\n02/09
VNT_Interval_Range                                                                                    
00:00-00:30          14.750000   15.750000       11.00       13.00       13.75       11.25        11.5
00:30-01:00          13.833333   15.583333        9.25       12.50       14.00       11.00        11.5
01:00-01:30          12.000000   13.250000        8.50        9.75        9.75        8.00         8.0


In [8]:
def generate_excel(vnt_b,vnt_a,pst_b,pst_a,pst_ivls,comparison,week_dates,week_monday,out_path):
    COL={'tb':'1F3864','db':'2E4057','bh':'2471A3','ah':'196F3D','dh':'6E2B8B',
         'ib':'D6DCE4','ic':'FFF2CC','bef':'BDD7EE','gn':'27AE60','ls':'C0392B',
         'nt':'EBEBEB','wt':'FFFFFF','dp':'1A8C4E','dn':'A93226','ch':'2C3E50','rc':'FFFDE7'}
    def fl(c): return PatternFill('solid',fgColor=c)
    def fn(bold=False,color='000000',sz=10): return Font(bold=bold,color=color,size=sz,name='Arial')
    def al(h='center',v='center',wrap=False): return Alignment(horizontal=h,vertical=v,wrap_text=wrap)
    sd=Side(style='thin',color='C0C0C0')
    def bd(): return Border(left=sd,right=sd,top=sd,bottom=sd)

    def ws_sum(ws,pvb,pva,title,ilbl,ilst,wdts,mul,dp=1):
        n=len(wdts); lc=1+n*3; last=get_column_letter(lc); fmt='0.0'
        ws.merge_cells(f'A1:{last}1')
        c=ws['A1']; c.value=title; c.font=fn(True,COL['wt'],13); c.fill=fl(COL['tb']); c.alignment=al()
        ws.row_dimensions[1].height=24
        ws.cell(2,1).value=ilbl; ws.cell(2,1).font=fn(True,COL['wt'],10)
        ws.cell(2,1).fill=fl(COL['tb']); ws.cell(2,1).alignment=al()
        for di,d in enumerate(wdts):
            col=2+di*3
            ws.merge_cells(f'{get_column_letter(col)}2:{get_column_letter(col+2)}2')
            c=ws.cell(2,col); c.value=d.strftime('%a\n%d/%m')
            c.font=fn(True,COL['wt'],10); c.fill=fl(COL['db']); c.alignment=al(wrap=True)
        ws.row_dimensions[2].height=30
        ws.cell(3,1).value='Interval'; ws.cell(3,1).font=fn(True,COL['wt'],9)
        ws.cell(3,1).fill=fl(COL['tb']); ws.cell(3,1).alignment=al()
        for di in range(n):
            col=2+di*3
            b=ws.cell(3,col);   b.value='Before'; b.fill=fl(COL['bh']); b.font=fn(True,COL['wt'],9); b.alignment=al()
            a=ws.cell(3,col+1); a.value='After';  a.fill=fl(COL['ah']); a.font=fn(True,COL['wt'],9); a.alignment=al()
            d=ws.cell(3,col+2); d.value='Delta';  d.fill=fl(COL['dh']); d.font=fn(True,COL['wt'],9); d.alignment=al()
        ws.row_dimensions[3].height=16; ws.freeze_panes='B4'
        dlk=[d.strftime('%a\n%d/%m') for d in wdts]
        for ri,ivl in enumerate(ilst,start=4):
            rch=False
            for di,dk in enumerate(dlk):
                col=2+di*3
                bvr=float(pvb.loc[ivl,dk]) if (ivl in pvb.index and dk in pvb.columns) else 0.0
                avr=float(pva.loc[ivl,dk]) if (ivl in pva.index and dk in pva.columns) else 0.0
                bv=round(bvr*mul,dp); av=round(avr*mul,dp); dv=round((avr-bvr)*mul,dp)
                if abs(dv)>0.001: rch=True
                bc=ws.cell(ri,col); bc.value=bv if bvr>0 else None
                bc.fill=fl(COL['bef']); bc.font=fn(sz=10); bc.alignment=al(); bc.border=bd(); bc.number_format=fmt
                ac=ws.cell(ri,col+1); ac.value=av if avr>0 else None
                ac.alignment=al(); ac.border=bd(); ac.number_format=fmt
                if dv>0.001:    ac.fill=fl(COL['gn']); ac.font=fn(sz=10,bold=True,color=COL['wt'])
                elif dv<-0.001: ac.fill=fl(COL['ls']); ac.font=fn(sz=10,bold=True,color=COL['wt'])
                else:           ac.fill=fl(COL['nt']); ac.font=fn(sz=10,color='888888')
                dc=ws.cell(ri,col+2); dc.value=dv if abs(dv)>0.001 else None
                dc.number_format=fmt; dc.alignment=al(); dc.border=bd()
                if dv>0.001:    dc.fill=fl(COL['dp']); dc.font=fn(bold=True,color=COL['wt'],sz=10)
                elif dv<-0.001: dc.fill=fl(COL['dn']); dc.font=fn(bold=True,color=COL['wt'],sz=10)
                else:           dc.fill=fl(COL['nt']); dc.font=fn(color='888888',sz=10)
            ic=ws.cell(ri,1); ic.value=ivl
            ic.fill=fl(COL['ic'] if rch else COL['ib'])
            ic.font=fn(bold=rch,sz=9); ic.alignment=al('left'); ic.border=bd()
            ws.row_dimensions[ri].height=14
        ws.column_dimensions['A'].width=14
        for di in range(n):
            col=2+di*3
            ws.column_dimensions[get_column_letter(col)].width=8.5
            ws.column_dimensions[get_column_letter(col+1)].width=8.5
            ws.column_dimensions[get_column_letter(col+2)].width=6.5

    def ws_det(ws,cmp,week_monday):
        DC=['Date_Converted','VNT_Interval_Range','PST_Interval_Range',
            'Productive_Before','Productive_After','Productive_Delta','Has_Change','Changed_Agents']
        DW=[13,14,14,13,13,10,10,55]
        co=cmp[[c for c in DC if c in cmp.columns]].copy()
        co['Date_Converted']=pd.to_datetime(co['Date_Converted']).dt.strftime('%Y-%m-%d')
        for c2 in ['Productive_Before','Productive_After','Productive_Delta']:
            if c2 in co.columns: co[c2]=co[c2].round(2)
        ws.merge_cells(f'A1:{get_column_letter(len(DC))}1')
        t=ws['A1']; t.value=f'Comparison Detail — Week {week_monday.strftime("%d %b %Y")}'
        t.font=fn(True,COL['wt'],12); t.fill=fl(COL['ch']); t.alignment=al()
        ws.row_dimensions[1].height=22
        hds=list(co.columns)
        for ci,h in enumerate(hds,start=1):
            c2=ws.cell(2,ci); c2.value=h; c2.font=fn(True,COL['wt'],10)
            c2.fill=fl(COL['ch']); c2.alignment=al(wrap=True); c2.border=bd()
        ws.row_dimensions[2].height=18; ws.freeze_panes='A3'
        for ri,row in enumerate(co.itertuples(index=False),start=3):
            isch=False
            for ci,val in enumerate(row,start=1):
                c2=ws.cell(ri,ci); c2.value=val; c2.border=bd(); c2.font=fn(sz=10)
                h=hds[ci-1]
                c2.alignment=al('left' if h in ('Date_Converted','VNT_Interval_Range','PST_Interval_Range','Changed_Agents') else 'center',wrap=(h=='Changed_Agents'))
                if h=='Has_Change' and val: isch=True
                if h=='Productive_Delta' and isinstance(val,(int,float)):
                    if val>0.001:    c2.fill=fl(COL['dp']); c2.font=fn(color=COL['wt'],bold=True,sz=10)
                    elif val<-0.001: c2.fill=fl(COL['dn']); c2.font=fn(color=COL['wt'],bold=True,sz=10)
            if isch:
                for ci2 in range(1,len(hds)+1):
                    if ws.cell(ri,ci2).fill.fgColor.rgb in ('00000000','FFFFFFFF','00FFFFFF'):
                        ws.cell(ri,ci2).fill=fl(COL['rc'])
            ws.row_dimensions[ri].height=14
        for ci,(h,w) in enumerate(zip(hds,DW[:len(hds)]+[12]*(len(hds)-len(DW))),start=1):
            ws.column_dimensions[get_column_letter(ci)].width=w

    wb=Workbook()
    wk=week_monday.strftime('%d %b %Y')
    ws1=wb.active; ws1.title='VNT Productive'
    ws_sum(ws1,vnt_b,vnt_a,f'Productive (VNT) — Week of {wk}','VNT Interval',ALL_VNT_INTERVALS,week_dates,mul=1,dp=1)
    ws2=wb.create_sheet('VNT Heads')
    ws_sum(ws2,vnt_b,vnt_a,f'Heads (VNT) — Week of {wk}','VNT Interval',ALL_VNT_INTERVALS,week_dates,mul=2,dp=1)
    ws3=wb.create_sheet('PST Productive')
    ws_sum(ws3,pst_b,pst_a,f'Productive (PST) — Week of {wk}','PST Interval',pst_ivls,week_dates,mul=1,dp=1)
    ws4=wb.create_sheet('PST Heads')
    ws_sum(ws4,pst_b,pst_a,f'Heads (PST) — Week of {wk}','PST Interval',pst_ivls,week_dates,mul=2,dp=1)
    ws5=wb.create_sheet('Detail')
    ws_det(ws5,comparison,week_monday)
    wb.save(out_path)
    print('Excel saved:',out_path)
    print('Sheets:',[s.title for s in wb.worksheets])

xl_path=os.path.join(PATHS['excel_output'],f'comparison_{WEEK_MONDAY}.xlsx')
generate_excel(vnt_b,vnt_a,pst_b,pst_a,ALL_PST_INTERVALS_ACTUAL,comparison,week_dates,_week,xl_path)


Excel saved: C:/Users/huuchinh.nguyen/Concentrix Corporation/WFM-Expedia-HCM - Branding files/Rawdata/INPUT_SCHEDULE/SCHEDULE_SIMULATION\comparison_2026-08-31.xlsx
Sheets: ['VNT Productive', 'VNT Heads', 'PST Productive', 'PST Heads', 'Detail']


In [9]:
CSS=(
    "<style>"
    ":root{--bef:#2980B9;--gn:#27AE60;--ls:#C0392B;--hi:#F39C12;"
    "--dp:#1A8C4E;--dn:#A93226;--ttl:#1F3864;--day:#2E4057;}"
    ".wrap{font-family:'Segoe UI',Arial,sans-serif;font-size:11px;overflow-x:auto;}"
    ".rpt-title{font-size:15px;font-weight:bold;color:var(--ttl);padding:8px 0 4px;"
    "border-bottom:3px solid var(--ttl);margin-bottom:8px;}"
    ".leg{display:flex;gap:14px;flex-wrap:wrap;margin-bottom:8px;font-size:11px;align-items:center;}"
    ".li{display:flex;align-items:center;gap:4px;}"
    ".lb{width:16px;height:16px;border-radius:3px;border:1px solid #999;}"
    "table{border-collapse:collapse;white-space:nowrap;}"
    "th{padding:4px 7px;border:1px solid #4a6278;text-align:center;"
    "font-size:10px;position:sticky;top:0;z-index:2;}"
    "th.td{background:var(--day);color:#fff;}"
    "th.tb{background:#2471A3;color:#fff;font-size:9px;font-weight:normal;}"
    "th.ta{background:#196F3D;color:#fff;font-size:9px;font-weight:normal;}"
    "th.tD{background:#6E2B8B;color:#fff;font-size:9px;font-weight:bold;}"
    "th.ti{background:var(--ttl);color:#fff;text-align:left;left:0;z-index:3;}"
    "td{padding:3px 6px;border:1px solid #dde;text-align:right;}"
    "td.iv{text-align:left;font-size:9px;font-family:monospace;background:#D6DCE4;"
    "position:sticky;left:0;z-index:1;min-width:110px;}"
    "td.iv.ch{background:#FFF2CC;border-left:3px solid var(--hi);font-weight:bold;}"
    "td.bef{background:#BDD7EE;color:#1A5276;}"
    "td.gn{background:var(--gn);color:#fff;font-weight:bold;}"
    "td.ls{background:var(--ls);color:#fff;font-weight:bold;}"
    "td.nt{background:#EBEBEB;color:#888;}"
    "td.zr{color:#ccc;}"
    "td.dp{background:var(--dp);color:#fff;font-weight:bold;}"
    "td.dn{background:var(--dn);color:#fff;font-weight:bold;}"
    "td.dz{background:#EBEBEB;color:#aaa;}"
    "</style>"
)

def render_html(pvb,pva,week_dates,week_monday,all_ivls,mul=1,dp=1,title_sfx="Productive"):
    dlk=[d.strftime("%a\n%d/%m") for d in week_dates]
    dhl=[d.strftime("%a<br>%d/%m") for d in week_dates]
    def fmt(v): return f"{v:.1f}"
    r1="<th class='ti' rowspan='2'>VNT Interval</th>"+"".join(f"<th class='td' colspan='3'>{x}</th>" for x in dhl)
    r2="".join("<th class='tb'>Before</th><th class='ta'>After</th><th class='tD'>Delta</th>" for _ in dlk)
    rows=""
    for ivl in all_ivls:
        has_ch=False; tds=""
        for dk in dlk:
            bvr=float(pvb.loc[ivl,dk]) if (ivl in pvb.index and dk in pvb.columns) else 0.0
            avr=float(pva.loc[ivl,dk]) if (ivl in pva.index and dk in pva.columns) else 0.0
            bv=round(bvr*mul,dp); av=round(avr*mul,dp); dv=round((avr-bvr)*mul,dp)
            if abs(dv)>0.001: has_ch=True
            bc="bef zr" if bvr==0 else "bef"
            ac="gn" if dv>0.001 else ("ls" if dv<-0.001 else ("nt zr" if avr==0 else "nt"))
            dc="dp" if dv>0.001 else ("dn" if dv<-0.001 else "dz")
            dtx=("+" if dv>0 else "")+fmt(dv) if abs(dv)>0.001 else ""
            tds+=(f"<td class='{bc}'>{fmt(bv) if bvr else ''}</td>"
                  f"<td class='{ac}'>{fmt(av) if avr else ''}</td>"
                  f"<td class='{dc}'>{dtx}</td>")
        ic="iv ch" if has_ch else "iv"
        rows+=f"<tr><td class='{ic}'>{ivl}</td>{tds}</tr>"
    ttl=f"<div class='rpt-title'>{title_sfx} (VNT) — Week of {week_monday.strftime('%d %b %Y')}</div>"
    leg=("<div class='leg'>"
         "<div class='li'><div class='lb' style='background:#BDD7EE'></div>Before</div>"
         "<div class='li'><div class='lb' style='background:#27AE60'></div>After Up</div>"
         "<div class='li'><div class='lb' style='background:#C0392B'></div>After Down</div>"
         "<div class='li'><div class='lb' style='background:#EBEBEB;border:1px solid #bbb'></div>No change</div>"
         "<div class='li'><div class='lb' style='background:#6E2B8B'></div>Delta per day</div>"
         "<div class='li'>Row yellow = changed</div>"
         "</div>")
    thead=f"<thead><tr>{r1}</tr><tr>{r2}</tr></thead>"
    return CSS+f"<div class='wrap'>{ttl}{leg}<table>{thead}<tbody>{rows}</tbody></table></div>"

display(HTML(render_html(vnt_b,vnt_a,week_dates,_week,ALL_VNT_INTERVALS,mul=1,dp=1,title_sfx="Productive")))


In [10]:
def parse_raw_iex_folder(folder_path, week_monday=None):
    import time as _time
    from datetime import datetime as _dt
    DROP_COLS  = ['__UNNAMED__4']
    RENAME_MAP = {
        'Agent Schedules':'Agent','__UNNAMED__1':'Date','__UNNAMED__2':'Start_Shift',
        '__UNNAMED__3':'End_Shift','__UNNAMED__5':'Scheduled Activity',
        '__UNNAMED__6':'Start_Action','__UNNAMED__9':'End_Action',
    }
    list_dfs=[]
    for fp in list(pathlib.Path(folder_path).glob('**/*.xlsx'))+list(pathlib.Path(folder_path).glob('**/*.csv')):
        if fp.name.startswith('~$'): continue
        export_dt=_dt(*_time.localtime(os.path.getmtime(fp))[:6])
        try:
            if HAS_POLARS:
                import polars as pl
                raw=(pl.read_excel(fp,infer_schema_length=0) if fp.suffix.lower()=='.xlsx'
                     else pl.read_csv(fp,infer_schema_length=0,encoding='utf-8',ignore_errors=True))
                df=raw.to_pandas()
            else:
                df=(pd.read_excel(fp,dtype=str) if fp.suffix.lower()=='.xlsx'
                    else pd.read_csv(fp,dtype=str,encoding='utf-8',errors='ignore'))
            df['sheet_name']=fp.stem; df['Export time']=export_dt
            list_dfs.append(df); print(f"  Read: {fp.name} ({len(df)} rows)")
        except Exception as ex:
            print(f"  Error: {fp.name} — {ex}")
    if not list_dfs: print("No files found."); return pd.DataFrame()
    IEX=pd.concat(list_dfs,ignore_index=True)
    IEX=IEX.drop(columns=[c for c in DROP_COLS if c in IEX.columns])
    IEX=IEX.rename(columns={k:v for k,v in RENAME_MAP.items() if k in IEX.columns})
    if 'Agent' not in IEX.columns:
        print("Column 'Agent' not found — check RENAME_MAP vs actual file columns:")
        print(list(IEX.columns[:10])); return IEX
    IEX['Generate Date']=np.where(IEX['Agent'].str.contains('Generation Date: ',na=False),
        IEX['Agent'].str.extract(r'Generation Date: (.+)')[0],np.nan)
    IEX['Generate Date']=IEX['Generate Date'].bfill()
    IEX['Agent']=IEX['Agent'].ffill()
    off_table=IEX[IEX['Start_Shift']=='Off'].copy()
    iex_edit=IEX[(IEX['Start_Shift']!='Off')&(IEX['Date']!='Date')&
                 ~(IEX['Date'].isna()&IEX['Scheduled Activity'].isna())].copy()
    iex_edit[['Date','Start_Shift','End_Shift']]=iex_edit[['Date','Start_Shift','End_Shift']].ffill()
    iex_edit=iex_edit[iex_edit['Agent'].str.contains('Agent: ',na=False)|iex_edit['Agent'].isna()]
    iex_edit=iex_edit[iex_edit['Scheduled Activity'].notna()]
    off_table['Scheduled Activity']=off_table['Scheduled Activity'].fillna(off_table['Start_Shift'])
    off_table['Start_Shift']=off_table['Start_Shift'].replace('Off',np.nan)
    result=pd.concat([off_table,iex_edit],axis=0,ignore_index=True)
    result['Date']=pd.to_datetime(result['Date'],errors='coerce')
    if 'Agent' in result.columns:
        result['IEX_ID']=result['Agent'].str.extract(r'(\d+)',expand=False).astype('Int64')
    else:
        result['IEX_ID']=pd.NA
    if week_monday is not None:
        wend=week_monday+pd.Timedelta(days=6)
        result=result[(result['Date']>=week_monday)&(result['Date']<=wend)]
    return result.sort_values(['Agent','Date','Start_Action'],na_position='first').reset_index(drop=True)

print("Parsing IEX_ACTUAL_ADJUSTMENT ...")
actual_files=list(pathlib.Path(PATHS['actual_adjustment']).glob('**/*.xlsx'))+list(pathlib.Path(PATHS['actual_adjustment']).glob('**/*.csv'))
if not actual_files:
    print("IEX_ACTUAL_ADJUSTMENT is empty.")
    print("Place the raw IEX export file(s) there after completing the swap in IEX system, then re-run this cell.")
    actual_structured=pd.DataFrame()
else:
    actual_structured=parse_raw_iex_folder(PATHS['actual_adjustment'],week_monday=_week)
    if actual_structured.empty:
        print('Parse returned empty DataFrame — check column debug output above.')
    elif 'IEX_ID' not in actual_structured.columns:
        print(f'Parsed {len(actual_structured):,} rows — but IEX_ID column missing. Check RENAME_MAP.')
        print('Columns found:', list(actual_structured.columns))
    else:
        n_agents = actual_structured['IEX_ID'].nunique()
        print(f'Parsed: {len(actual_structured):,} rows | {n_agents} agents')

Parsing IEX_ACTUAL_ADJUSTMENT ...
  Error: 20226_08_31.xlsx — name 'HAS_POLARS' is not defined
No files found.
Parse returned empty DataFrame — check column debug output above.


In [11]:
def compare_planned_vs_actual(planned_path, actual_df, week_monday):
    if not os.path.exists(planned_path):
        print(f"Planned file not found: {planned_path}"); return pd.DataFrame(), pd.DataFrame()
    if actual_df.empty:
        print("Actual data is empty."); return pd.DataFrame(), pd.DataFrame()
    week_end=week_monday+pd.Timedelta(days=6)
    planned=pd.read_csv(planned_path,dtype=str)
    planned['Date']=pd.to_datetime(planned['Date'],errors='coerce')
    planned['IEX_ID']=planned['Agent'].str.extract(r'(\d+)',expand=False).astype('Int64')
    planned=planned[(planned['Date']>=week_monday)&(planned['Date']<=week_end)].copy()
    actual=actual_df[(actual_df['Date']>=week_monday)&(actual_df['Date']<=week_end)].copy()
    for df in [planned,actual]:
        df['_key']=(df['IEX_ID'].astype(str)+'|'+df['Date'].astype(str)+'|'+
                    df['Scheduled Activity'].fillna('')+'|'+df['Start_Action'].fillna('')+'|'+df['End_Action'].fillna(''))
    pk,ak=set(planned['_key']),set(actual['_key'])
    po=planned[planned['_key'].isin(pk-ak)].copy(); po['Status']='Planned Only (missing from Actual)'
    ao=actual[actual['_key'].isin(ak-pk)].copy();   ao['Status']='Actual Only (extra vs Planned)'
    SHOW=['IEX_ID','Agent','Date','Scheduled Activity','Start_Shift','End_Shift','Start_Action','End_Action','Status']
    diff=pd.concat([po[[c for c in SHOW if c in po.columns]],
                    ao[[c for c in SHOW if c in ao.columns]]],ignore_index=True)
    diff=diff.sort_values(['IEX_ID','Date','Start_Action'],na_position='first')
    summary=(diff.groupby(['IEX_ID','Date','Status'],sort=False)
             .agg(Activities=('Scheduled Activity','count')).reset_index())
    print(f"Planned Only (missing from Actual): {len(po):,} rows")
    print(f"Actual Only  (extra vs Planned)   : {len(ao):,} rows")
    print(f"Matched rows                       : {len(pk&ak):,}")
    return diff,summary

ws_str=_week.strftime('%Y_%m_%d')
planned_adj_path=str(next((p for p in pathlib.Path(PATHS['after_adjustment']).glob(f'IEX_Adjusted_{ws_str}.csv')),pathlib.Path('')))

if actual_structured.empty:
    print("No actual data — skipping Planned vs Actual comparison.")
    diff_df=pd.DataFrame(); summary_df=pd.DataFrame()
else:
    diff_df,summary_df=compare_planned_vs_actual(planned_adj_path,actual_structured,_week)

No actual data — skipping Planned vs Actual comparison.


In [12]:
if not diff_df.empty:
    ACSS="""<style>
.at{font-family:'Segoe UI',Arial,sans-serif;font-size:11px;overflow-x:auto;}
.at-title{font-size:14px;font-weight:bold;color:#1F3864;padding:6px 0 4px;border-bottom:3px solid #1F3864;margin-bottom:8px;}
.at-stat{background:#EBF5FB;border:1px solid #AED6F1;padding:6px 12px;border-radius:4px;margin-bottom:8px;}
table.at-tbl{border-collapse:collapse;width:100%;}
table.at-tbl th{background:#2C3E50;color:#fff;padding:4px 8px;border:1px solid #4a6278;font-size:10px;}
table.at-tbl td{padding:3px 7px;border:1px solid #dde;font-size:10px;}
tr.po{background:#FADBD8;} tr.ao{background:#D5F5E3;}
td.sp{color:#C0392B;font-weight:bold;} td.sa{color:#27AE60;font-weight:bold;}
</style>"""
    title_html=f"<div class='at-title'>Actual vs Planned Audit — Week of {_week.strftime('%d %b %Y')}</div>"
    po_n=(diff_df['Status'].str.contains('Planned Only')).sum()
    ao_n=(diff_df['Status'].str.contains('Actual Only')).sum()
    stat_html=(f"<div class='at-stat'>"
               f"<b>Planned Only</b> (rows in IEX_AFTER_ADJUSTMENT missing from Actual): "
               f"<span style='color:#C0392B;font-weight:bold'>{po_n}</span> rows &nbsp;|&nbsp;"
               f"<b>Actual Only</b> (rows in Actual not in Planned): "
               f"<span style='color:#27AE60;font-weight:bold'>{ao_n}</span> rows"
               f"</div>")
    show_cols=[c for c in ['IEX_ID','Agent','Date','Scheduled Activity','Start_Shift','End_Shift','Start_Action','End_Action','Status'] if c in diff_df.columns]
    rows_html=''
    for _,r in diff_df.head(300).iterrows():
        s=str(r.get('Status',''))
        tr_cls='po' if 'Planned' in s else 'ao'
        td_cls='sp' if 'Planned' in s else 'sa'
        tds=''.join(f'<td {"class=\\'"+td_cls+"\\' " if c=="Status" else ""}>{str(r.get(c,""))}</td>' for c in show_cols)
        rows_html+=f"<tr class='{tr_cls}'>{tds}</tr>"
    hdrs=''.join(f'<th>{c}</th>' for c in show_cols)
    table_html=f"<table class='at-tbl'><thead><tr>{hdrs}</tr></thead><tbody>{rows_html}</tbody></table>"
    if len(diff_df)>300:
        table_html+=f"<p style='color:#888;font-style:italic'>Showing 300 of {len(diff_df)} rows.</p>"
    display(HTML(ACSS+f"<div class='at'>{title_html}{stat_html}{table_html}</div>"))
else:
    print("No diff data. Place IEX export in IEX_ACTUAL_ADJUSTMENT folder and run Cells 10-11.")

No diff data. Place IEX export in IEX_ACTUAL_ADJUSTMENT folder and run Cells 10-11.
